<a href="https://colab.research.google.com/github/misbah2611hsn/Portfolio/blob/main/Lora_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install peft

In [2]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 12.8 MB/s eta 0:00:00


In [3]:
!pip install torch

In [4]:
from google.colab import userdata
from huggingface_hub import login

# Login into Hugging Face Hub
hf_token = userdata.get('tok1') # If you are running inside a Google Colab
login(hf_token)

In [5]:
from datasets import load_dataset

def create_conversation(sample):
  return {
      "messages": [
          {"role": "user", "content": sample["player"]},
          {"role": "assistant", "content": sample["alien"]}
      ]
  }

npc_type = "martian"

# Load dataset from the Hub
dataset = load_dataset("bebechien/MobileGameNPC", npc_type, split="train")

# Convert dataset to conversational format
dataset[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/141 [00:00<?, ?B/s]

martian.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/25 [00:00<?, ? examples/s]

{'id': 1,
 'category': 'Greetings & Basic Questions',
 'player': 'Hello there.',
 'alien': "Gree-tongs, Terran. You'z a long way from da Blue-Sphere, yez?"}

In [6]:
dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)

# Split dataset into 80% training samples and 20% test samples
dataset = dataset.train_test_split(test_size=0.2, shuffle=False)

# Print formatted user prompt
print(dataset["train"][0]["messages"])

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

[{'content': 'Hello there.', 'role': 'user'}, {'content': "Gree-tongs, Terran. You'z a long way from da Blue-Sphere, yez?", 'role': 'assistant'}]


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID="google/gemma-3-270m-it"
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,

)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,              # more stable for Gemma-3 in Colab
)
input_text = "Write me a poem about Machine Learning."
input_ids = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**input_ids)
print(tokenizer.decode(outputs[0]))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<bos>Write me a poem about Machine Learning.

A digital eye, a silent hum,
Learning quickly, without a glum.
It sees


In [8]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Keep sequences modest; TRL respects this limit
tokenizer.model_max_length = 512

In [9]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"] # make sure to save the lm_head and embed_tokens as you train the special tokens
)

In [10]:
from trl import SFTConfig

args = SFTConfig(
    output_dir="gemmaLorra001",         # directory to save and repository id
    # max_seq_length=512,                     # max sequence length for model and packing of the dataset
    packing=True,                           # Groups multiple samples in the dataset into a single sequence
    num_train_epochs=3,                     # number of training epochs
    per_device_train_batch_size=8,          # batch size per device during training
    gradient_accumulation_steps=4,          # number of steps before performing a backward/update pass
    gradient_checkpointing=True,            # use gradient checkpointing to save memory
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=10,                       # log every 10 steps
    save_strategy="epoch",                  # save checkpoint every epoch
    learning_rate=2e-4,
                                             # learning rate, based on QLoRA paper
    # fp16=True if torch_dtype == torch.float16 else False,   # use float16 precision
    # bf16=True if torch_dtype == torch.bfloat16 else False,   # use bfloat16 precision
    max_grad_norm=0.3,                      # max gradient norm based on QLoRA paper
    warmup_ratio=0.03,                      # warmup ratio based on QLoRA paper
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    push_to_hub=True,                       # push model to hub
    report_to="tensorboard",                # report metrics to tensorboard
    dataset_kwargs={
        "add_special_tokens": False, # Template with special tokens
        "append_concat_token": True, # Add EOS token as separator token between examples
    }
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
from trl import SFTTrainer

# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

In [12]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss


TrainOutput(global_step=3, training_loss=6.990383783976237, metrics={'train_runtime': 120.7899, 'train_samples_per_second': 0.05, 'train_steps_per_second': 0.025, 'total_flos': 5080089318912.0, 'train_loss': 6.990383783976237})

In [13]:
from sklearn.metrics import accuracy_score
import torch

def evaluate_model(model, dataset, tokenizer, max_samples=None):
    model.eval()
    correct = 0
    total = 0

    eval_data = dataset["test"]
    if max_samples:
        eval_data = eval_data.select(range(min(max_samples, len(eval_data))))

    with torch.no_grad():
        for example in eval_data:
            messages = example["messages"]
            text = tokenizer.apply_chat_template(messages, tokenize=False)
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

            # Move input_ids to the same device as the model
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            outputs = model.generate(**inputs, max_new_tokens=50)
            predicted = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Compare predicted response with expected
            expected = messages[-1]["content"]  # Last message is assistant response
            if expected.lower() in predicted.lower():
                correct += 1
            total += 1

    accuracy = correct / total if total > 0 else 0
    print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
    return accuracy

# Run evaluation
evaluate_model(model, dataset, tokenizer, max_samples=100)

Accuracy: 100.00% (5/5)


1.0

In [14]:
trainer.model.save_pretrained("gemma-270-lora-finetuned06032026")
tokenizer.save_pretrained("gemma-270-lora-finetuned06032026")

('gemma-270-lora-finetuned06032026/tokenizer_config.json',
 'gemma-270-lora-finetuned06032026/chat_template.jinja',
 'gemma-270-lora-finetuned06032026/tokenizer.json')

In [15]:
import torch
from transformers import pipeline

model_path = "gemma-270-lora-finetuned06032026"

# Load Model with PEFT adapter
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Test the model with an input prompt
input_text = "Hello There."
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# Generate predictions
outputs = model.generate(input_ids, max_length=50, num_return_sequences=1, temperature=0.7)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Generated Text:")
print(generated_text)

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Gemma3ForCausalLM LOAD REPORT from: gemma-270-lora-finetuned06032026
Key                                               | Status     | 
--------------------------------------------------+------------+-
base_model.model.lm_head.weight                   | UNEXPECTED | 
model.embed_tokens.weight                         | UNEXPECTED | 
lm_head.modules_to_save.default.weight            | MISSING    | 
model.embed_tokens.modules_to_save.default.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
The attention mask is not set and cannot

Generated Text:
Hello There.

Hello, anyone?

Hello, what is your name?

Hello, what kind of news are you looking for?

Hello, what is your favorite thing to do?

Hello, what kind of music do you like


In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# The model saved in "gemma-270-lora-finetuned06032026" is already the fine-tuned model
# with the LoRA adapters and modules_to_save (like embed_tokens) incorporated.
# Therefore, we load it directly using AutoModelForCausalLM, as no further explicit
# merging is necessary or correctly handled by AutoPeftModelForCausalLM in this case.
model_path = "gemma-270-lora-finetuned06032026"

# Load the fine-tuned model and its tokenizer
merged_model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Save the model and tokenizer again under a new name if desired.
# This step is mostly for creating a new directory with a different name,
# but the content is essentially the fully fine-tuned model from model_path.
merged_model.save_pretrained("gemma-merged-final06032026")
tokenizer.save_pretrained("gemma-merged-final06032026")

print(f"Model successfully loaded and saved as a merged model in 'gemma-merged-final06032026'.")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Gemma3ForCausalLM LOAD REPORT from: gemma-270-lora-finetuned06032026
Key                                               | Status     | 
--------------------------------------------------+------------+-
base_model.model.lm_head.weight                   | UNEXPECTED | 
model.embed_tokens.weight                         | UNEXPECTED | 
lm_head.modules_to_save.default.weight            | MISSING    | 
model.embed_tokens.modules_to_save.default.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully loaded and saved as a merged model in 'gemma-merged-final06032026'.
